# Qdrant Local File-Based Vector Store

This notebook demonstrates FFAI's Qdrant backend in **local mode** —
data is persisted to disk in a directory you choose.

- No server required (embedded Qdrant via `qdrant-client`)
- Data survives across kernel restarts
- Uses a temporary directory that is cleaned up at the end

We use synthetic embeddings so no external embedding service is needed.

<div class="page-break"></div>

---

## Section 1: Setup

In [ ]:
import sys
import tempfile
from pathlib import Path

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

import numpy as np  # noqa: E402

from ffai.rag.stores import get_store  # noqa: E402

DB_DIR = Path(tempfile.mkdtemp(prefix='qdrant_local_demo_'))

print(f'Project root: {_project_root}')
print(f'Storage directory: {DB_DIR}')

<div class="page-break"></div>

---

## Section 2: Create the Store

Pass `path=` to use local file storage. The directory is created automatically.

In [ ]:
DIM = 128

store = get_store(
    backend='qdrant',
    collection_name='demo_local',
    embedding_dim=DIM,
    path=str(DB_DIR),
)

print(f'Backend: {store.name}')
print(f'Storage: {DB_DIR}')
print(f'Files on disk: {list(DB_DIR.iterdir())}')

<div class="page-break"></div>

---

## Section 3: Add and Index Documents

Load the sample documents from `examples/_rag_data/`, chunk them, and
add with synthetic embeddings.

In [ ]:
from examples._rag_data.download import download

docs = download()
for name, p in docs.items():
    print(f'  {name}: {p.stat().st_size:,} bytes')

In [ ]:
def make_embedding(text: str, dim: int = DIM) -> list[float]:
    rng = np.random.default_rng(hash(text) & 0xFFFFFFFF)
    vec = rng.standard_normal(dim)
    return (vec / np.linalg.norm(vec)).tolist()


from ffai.rag._async import run_sync  # noqa: E402
from ffai.rag.splitters import get_chunker  # noqa: E402

total = 0
for name, path in docs.items():
    text = path.read_text(encoding='utf-8')
    chunks = get_chunker('markdown', chunk_size=400, chunk_overlap=40).chunk(text)
    ids = [f'{name}_chunk{i}' for i in range(len(chunks))]
    texts = [c.content for c in chunks]
    embs = [make_embedding(t) for t in texts]
    metas = [{'source': name, 'chunking_strategy': 'markdown'} for _ in chunks]
    added = run_sync(store.aadd(ids, texts, embs, metas))
    total += added
    print(f'  {name}: {added} chunks')

print(f'\nTotal indexed: {total} chunks')
print(f'Sources: {store.list_sources()}')

<div class="page-break"></div>

---

## Section 4: Verify Persistence

Re-open the store from the same directory. The data should still be there.

In [ ]:
saved_count = store.count()
store._client.close()

store2 = get_store(
    backend='qdrant',
    collection_name='demo_local',
    embedding_dim=DIM,
    path=str(DB_DIR),
)

print(f'Re-opened store count: {store2.count()}')
print(f'Sources: {store2.list_sources()}')
print(f'Same data: {store2.count() == saved_count}')

<div class="page-break"></div>

---

## Section 5: Search with Metadata Filters

Search within a single source document.

In [ ]:
query_text = 'How do I authenticate with a bearer token?'
query_emb = make_embedding(query_text)

hits = run_sync(store2.asearch(
    query_emb, top_k=3, where={'source': 'api_docs.md'},
))

print(f'Query: {query_text}')
print('Filter: source=api_docs.md')
print(f'Results: {len(hits)}\n')
for i, h in enumerate(hits, 1):
    print(f'  {i}. [{h.score:.4f}] {h.content[:80]}...')
    print(f'     source={h.source}')
    print()

<div class="page-break"></div>

---

## Section 6: Cleanup

Clear the collection and remove the temporary storage directory.

In [ ]:
import shutil

store2.clear()
print(f'After clear: {store2.count()} chunks')

if DB_DIR.exists():
    shutil.rmtree(DB_DIR)
    print(f'Removed directory: {DB_DIR}')
else:
    print(f'Directory already gone: {DB_DIR}')

print('\nCleanup complete. No files left on disk.')